# SVM Model

Train and evaluate a Support Vector Machine classifier.

In [ ]:
# Check if running in Google Colab and set up environment
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    print("Running in Google Colab. Cloning repository...")
    !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
    %cd Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
import os
import numpy as np
import joblib
import pandas as pd
from pathlib import Path

DATA_DIR = Path("./processed_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not os.path.exists("./processed_data/X_hog.npy") or not os.path.exists("./processed_data/y_labels.npy") or not os.path.exists("./processed_data/label_mapping.pkl"):
    print("Processed feature files not found. Generating them using the aligned 20-species subset...")
    
    if not os.path.exists("metadata_preprocessed.csv"):
        print("metadata_preprocessed.csv not found! Running dataset preparation setup...")
        raise FileNotFoundError("metadata_preprocessed.csv is required. Please run notebooks/01_Data_Preparation.ipynb first.")
        
    metadata_df = pd.read_csv("metadata_preprocessed.csv")
    
    import sys
    sys.path.append(str(Path(".").resolve()))
    from src.feature_extraction import extract_hog_features
    from tqdm.auto import tqdm
    
    X_hog = []
    y = []
    
    print("Extracting HOG features from preprocessed images...")
    for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
        img_path = Path(row["preprocessed_image_path"])
        feat = extract_hog_features(img_path, target_size=(128, 128))
        X_hog.append(feat)
        y.append(row["class_id"] - 1)
        
    X_hog = np.array(X_hog, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    
    unique_classes = sorted(metadata_df["class_name"].unique())
    label_mapping = {class_name: idx for idx, class_name in enumerate(unique_classes)}
    
    np.save(DATA_DIR / "X_hog.npy", X_hog)
    np.save(DATA_DIR / "y_labels.npy", y)
    joblib.dump(label_mapping, DATA_DIR / "label_mapping.pkl")
    print("HOG feature vectors generated and saved to ./processed_data successfully!")
else:
    print("Processed feature files already exist. Ready to train!")

## Setup & Load Saved HOG Features

In [ ]:
import os
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

DATA_DIR = "./processed_data"

X_hog = np.load(os.path.join(DATA_DIR, "X_hog.npy"))
y_labels = np.load(os.path.join(DATA_DIR, "y_labels.npy"))
label_mapping = joblib.load(os.path.join(DATA_DIR, "label_mapping.pkl"))

print(f"Loaded HOG Feature Matrix X: {X_hog.shape}")
print(f"Loaded Target Labels y: {y_labels.shape}")
print(f"Number of classes: {len(label_mapping)}")

## Stratified Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_hog,
    y_labels,
    test_size=0.20,
    random_state=42,
    stratify=y_labels
)

print(f"Training Samples: {X_train.shape[0]}")
print(f"Testing Samples : {X_test.shape[0]}")

## Train Baseline SVM Classifier

In [ ]:
print("--- Training Baseline SVM Classifier ---")

svm_baseline = SVC(kernel='rbf', C=1.0, random_state=42)
svm_baseline.fit(X_train, y_train)

y_pred_base = svm_baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred_base)

print(f"\nBaseline SVM Test Accuracy: {baseline_acc * 100:.2f}%")

## Hyperparameter Tuning

In [ ]:
print("--- Hyperparameter Tuning with GridSearchCV ---")

param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.001, 0.01]
}

grid_search = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\n--- Tuning Results ---")
print(f"Best Hyperparameters : {grid_search.best_params_}")
print(f"Best Cross-Val Score : {grid_search.best_score_ * 100:.2f}%")

## Evaluate & Save Best SVM Model

In [ ]:
best_svm = grid_search.best_estimator_
y_pred = best_svm.predict(X_test)
final_acc = accuracy_score(y_test, y_pred)

print(f"\nFinal Tuned SVM Test Accuracy: {final_acc * 100:.2f}%")

MODELS_DIR = "./models"
os.makedirs(MODELS_DIR, exist_ok=True)

model_path = os.path.join(MODELS_DIR, "svm_model.pkl")
joblib.dump(best_svm, model_path)
print(f"Best SVM model saved to '{model_path}' successfully!")

class_names = [k.split('.')[-1].replace('_', ' ') for k in sorted(label_mapping, key=label_mapping.get)]
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names))

## Confusion Matrix Visualization

In [ ]:
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)

plt.title("Confusion Matrix - Tuned SVM")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()